In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from config import TOKENIZED_DIR, FEATURE_DIR

# Step 11
class ClipCapDataset(Dataset):
    def __init__(self, tokenized_path, clip_feature_data):
        self.tokenized_data = torch.load(tokenized_path)
        self.image_ids = self.tokenized_data["image_ids"] #list
        self.input_ids = self.tokenized_data["input_ids"] #string
        self.attention_mask = self.tokenized_data["attention_mask"] #string
        
        # matrix các vector feature ảnh
        self.clip_features = clip_feature_data["features"]
        
        # hashmap: img_id (name of img) -> idx
        self.img_id_to_idx = {
            img_id: idx 
            for idx, img_id in enumerate(clip_feature_data["image_ids"])
        }

        # validation
        token_image_ids = set(self.image_ids)
        clip_image_ids = set(clip_feature_data["image_ids"])

        assert len(clip_feature_data["image_ids"]) == len(clip_image_ids), \
            "Duplicate image_id in CLIP features"

        missing_clip = token_image_ids - clip_image_ids
        if missing_clip:
            raise ValueError(
                f"Missing CLIP features for {len(missing_clip)} images. "
                f"Examples: {list(missing_clip)[:5]}"
            )

        print(
            f"Alignment PASS: "
            f"{len(token_image_ids)} images ↔ {len(self.image_ids)} caption samples"
        )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        feature_idx = self.img_id_to_idx[img_id]
        image_embed = self.clip_features[feature_idx]
        input_ids = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100 # gán -100 cho các vị trí đệm để hàm CrossEntropyLoss skip
        
        return {
            "image_embed": image_embed,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

d:\Zfs_Clip_Image\zfs-clip-image-captioning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 12
def create_dataloaders(batch_size=32, train_filename="train.pt"):
    clip_feature_path = FEATURE_DIR / "clip_features.pt"
    clip_feature_data = torch.load(clip_feature_path)
    
    dataloaders = {}

    train_path = TOKENIZED_DIR / train_filename
    train_dataset = ClipCapDataset(train_path, clip_feature_data)
    val_dataset = ClipCapDataset(TOKENIZED_DIR / "val.pt", clip_feature_data)
    test_dataset = ClipCapDataset(TOKENIZED_DIR / "test.pt", clip_feature_data)

    # check data leakage (intersection of any two sets must be empty)
    train_ids = set(train_dataset.image_ids)
    val_ids = set(val_dataset.image_ids)
    test_ids = set(test_dataset.image_ids)

    assert train_ids.isdisjoint(val_ids), \
        "Data leakage: Train and Validation share images"
    assert train_ids.isdisjoint(test_ids), \
        "Data leakage: Train and Test share images"
    assert val_ids.isdisjoint(test_ids), \
        "Data leakage: Validation and Test share images"

    # split infos
    all_ids = train_ids | val_ids | test_ids
    print(f"Train images : {len(train_ids)}")
    print(f"Val images   : {len(val_ids)}")
    print(f"Test images  : {len(test_ids)}")
    print(f"Total images : {len(all_ids)}")

    dataloaders["train"] = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    dataloaders["val"] = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    dataloaders["test"] = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return dataloaders

In [3]:
#Testing
loaders = create_dataloaders(batch_size=32)

print("Check shape của batch 1")
sample_batch = next(iter(loaders["train"]))

print(f"Image Embed Shape  : {sample_batch['image_embed'].shape}")
print(f"Input IDs Shape    : {sample_batch['input_ids'].shape}")
print(f"Attention Mask     : {sample_batch['attention_mask'].shape}")
print(f"Labels Shape       : {sample_batch['labels'].shape}")

print("\n• Sample Labels:")
print(sample_batch['labels'][0])

Alignment PASS: 6462 images ↔ 32310 caption samples
Alignment PASS: 807 images ↔ 4035 caption samples
Alignment PASS: 809 images ↔ 4045 caption samples
Train images : 6462
Val images   : 807
Test images  : 809
Total images : 8078
Check shape của batch 1
Image Embed Shape  : torch.Size([32, 512])
Input IDs Shape    : torch.Size([32, 48])
Attention Mask     : torch.Size([32, 48])
Labels Shape       : torch.Size([32, 48])

• Sample Labels:
tensor([26829,   389, 39153,   287,   262,  7463,  1660,   764,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100])
